# Trabalho Final - LLM e Deep Learning

Aluna: Giovanna Oliveira Martins

## Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import requests
import re
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter, defaultdict
import tiktoken
import json
import random

from transformers import GPT2Config, GPT2LMHeadModel

from scipy.stats import shapiro, ttest_rel, wilcoxon

from sacrebleu.metrics import BLEU

## Download do Tiny Shakespeare

In [2]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

text = requests.get(url).text

print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



## Tokenização

In [3]:
tokenizer = tiktoken.get_encoding("gpt2")

In [4]:
tokens = tokenizer.encode(text)
texto = tokenizer.decode(tokens)

In [5]:
vocab_size = tokenizer.n_vocab

print(vocab_size)

50257


In [6]:
def encode(text):
    return tokenizer.encode(text)

def decode(tokens):
    return tokenizer.decode(tokens)

In [7]:
data = torch.tensor(
    encode(text),
    dtype=torch.long
)

## Separação 
90% treino | 5% validação | 5% teste

In [ ]:
train_ratio = 0.90
val_ratio = 0.05

train_end = int(len(data) * train_ratio)
val_end = int(len(data) * (train_ratio + val_ratio))

train_data = data[:train_end]
val_data = data[train_end:val_end]
test_data = data[val_end:]

print(f"Total de tokens : {len(data)}")
print(f"Treino          : {len(train_data)}")
print(f"Validação       : {len(val_data)}")
print(f"Teste           : {len(test_data)}")

Total de tokens : 338025
Treino          : 304222
Validação       : 16901
Teste           : 16902


## Dataset e DataLoader

In [9]:
batch_size = 32
block_size = 128

In [10]:
class ShakespeareDataset(Dataset):

    def __init__(self, data, block_size):

        self.block_size = block_size

        self.samples = []

        for i in range(0, len(data)-block_size-1, block_size):

            x = data[i:i+block_size]

            y = data[i+1:i+block_size+1]

            self.samples.append((x, y))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

In [11]:
train_dataset = ShakespeareDataset(
    train_data,
    block_size
)

val_dataset = ShakespeareDataset(
    val_data,
    block_size
)

test_dataset = ShakespeareDataset(
    test_data,
    block_size
)

In [12]:
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [13]:
x, y = next(iter(train_loader))

print(x.shape)
print(y.shape)

torch.Size([32, 128])
torch.Size([32, 128])


In [14]:
sample = x[0]

print(decode(sample.tolist()))

 of gentler, milder mould.

KATHARINA:
I'faith, sir, you shall never need to fear:
I wis it is not half way to her heart;
But if it were, doubt not her care should be
To comb your noddle with a three-legg'd stool
And paint your face and use you like a fool.

HORTENSIA:
From all such devils, good Lord deliver us!

GREMIO:
And me too, good Lord!

TRANIO:
Hush, master! here's some good


## Modelo

In [15]:
d_model = 256
n_head = 4
n_layer = 4
dropout = 0.1

In [ ]:
config = GPT2Config(
    vocab_size=vocab_size,
    n_positions=block_size,           # tamanho máximo da sequência
    n_embd=d_model,
    n_layer=n_layer,
    n_head=n_head,                
    resid_pdrop=dropout,
    embd_pdrop=dropout,
    attn_pdrop=dropout
)

model = GPT2LMHeadModel(config)

## Treino

In [17]:
learning_rate = 3e-4
num_epochs = 50
patience = 5

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate
)

n_params = sum(
    p.numel()
    for p in model.parameters()
)

print(f"Número de parâmetros: {n_params}")

Número de parâmetros: 16058112


In [19]:
def train_epoch(model, dataloader, optimizer, device):

    model.train()

    total_loss = 0.0

    for x, y in dataloader:

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        outputs = model(input_ids=x)

        logits = outputs.logits

        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            y.view(-1)
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [20]:
@torch.no_grad()
def validate_epoch(model, dataloader, device):

    model.eval()

    total_loss = 0.0

    for x, y in dataloader:

        x = x.to(device)
        y = y.to(device)

        outputs = model(input_ids=x)

        logits = outputs.logits

        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            y.view(-1)
        )

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [21]:
best_val_loss = float("inf")
epochs_without_improvement = 0

for epoch in range(num_epochs):

    train_loss = train_epoch(
        model,
        train_loader,
        optimizer,
        device
    )

    val_loss = validate_epoch(
        model,
        val_loader,
        device
    )

    print(
        f"Epoch {epoch+1:03d}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        epochs_without_improvement = 0

        torch.save(
            model.state_dict(),
            "best_model.pt"
        )

        print(">> Melhor modelo salvo.")

    else:

        epochs_without_improvement += 1

        print(
            f">> Sem melhora "
            f"({epochs_without_improvement}/{patience})"
        )

    if epochs_without_improvement >= patience:

        print("\nEarly stopping acionado.")
        break

Epoch 001/50 | Train Loss: 7.5604 | Val Loss: 6.1543
>> Melhor modelo salvo.
Epoch 002/50 | Train Loss: 5.9131 | Val Loss: 5.6571
>> Melhor modelo salvo.
Epoch 003/50 | Train Loss: 5.3990 | Val Loss: 5.2346
>> Melhor modelo salvo.
Epoch 004/50 | Train Loss: 5.0165 | Val Loss: 4.9873
>> Melhor modelo salvo.
Epoch 005/50 | Train Loss: 4.7765 | Val Loss: 4.8393
>> Melhor modelo salvo.
Epoch 006/50 | Train Loss: 4.5901 | Val Loss: 4.7241
>> Melhor modelo salvo.
Epoch 007/50 | Train Loss: 4.4351 | Val Loss: 4.6334
>> Melhor modelo salvo.
Epoch 008/50 | Train Loss: 4.3010 | Val Loss: 4.5534
>> Melhor modelo salvo.
Epoch 009/50 | Train Loss: 4.1858 | Val Loss: 4.4907
>> Melhor modelo salvo.
Epoch 010/50 | Train Loss: 4.0753 | Val Loss: 4.4521
>> Melhor modelo salvo.
Epoch 011/50 | Train Loss: 3.9811 | Val Loss: 4.4337
>> Melhor modelo salvo.
Epoch 012/50 | Train Loss: 3.8932 | Val Loss: 4.4164
>> Melhor modelo salvo.
Epoch 013/50 | Train Loss: 3.7993 | Val Loss: 4.4357
>> Sem melhora (1/5)
Ep

In [22]:
model.load_state_dict(
    torch.load(
        "best_model.pt",
        map_location=device
    )
)

model.eval()

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 256)
    (wpe): Embedding(128, 256)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-3): 4 x GPT2Block(
        (ln_1): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=768, nx=256)
          (c_proj): Conv1D(nf=256, nx=256)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=1024, nx=256)
          (c_proj): Conv1D(nf=256, nx=1024)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=256, out_features=50257, bias=False)
)

## Geração

### Top-k Sampling

Seleciona o próximo token usando Top-k Sampling.

In [33]:
def top_k_sampling(logits, k=50):

    k = min(k, logits.size(-1))

    values, indices = torch.topk(logits, k)

    probs = torch.softmax(values, dim=-1)

    sampled = torch.multinomial(probs, num_samples=1)

    next_token = indices[sampled]

    return next_token.item()

### Presence Penalty

Penaliza apenas a presença do token.

In [24]:
def apply_presence_penalty(
    logits,
    generated_tokens,
    alpha=0.5
):
    
    logits = logits.clone()

    used = set(generated_tokens)

    for token in used:
        logits[token] -= alpha

    return logits

### Frequency Penalty

Penalização proporcional à frequência (cresce conforme a frequência).

In [ ]:
def apply_frequency_penalty(
    logits,
    generated_tokens,
    beta=0.5
):

    logits = logits.clone()

    counts = Counter(generated_tokens)

    for token, freq in counts.items():

        logits[token] -= beta * freq

    return logits

### Penalização Homeostática

Cada token possui um estado interno Hi​ que varia continuamente. Sempre que o token aparece Hi​ aumenta. Em cada passo todos sofrem decaimento exponencial.

Penalização proporcional à Hi (estado homeostático).


In [26]:
class Homeostasis:

    def __init__(
        self,
        decay=0.95,
        increment=1.0
    ):

        self.decay = decay
        self.increment = increment

        self.state = defaultdict(float)

    def decay_all(self):

        for token in list(self.state.keys()):

            self.state[token] *= self.decay

    def update(self, token):

        self.state[token] += self.increment

    def penalty(self, token):

        return self.state[token]

In [27]:
def apply_homeostatic_penalty(
    logits,
    homeostasis,
    gamma=0.5
):

    logits = logits.clone()

    for token in homeostasis.state:

        logits[token] -= gamma * homeostasis.penalty(token)

    return logits

### Geração Autoregressiva

In [ ]:
def generate(
    model,
    prompt,
    strategy="baseline",
    max_new_tokens=100,
    top_k=50,
    alpha=0.5,
    beta=0.5,
    gamma=0.5,
    decay=0.95,
    increment=1.0,
    device="cpu",
    seed=None
):

    model.eval()

    if seed is not None:
        random.seed(seed)
        torch.manual_seed(seed)

        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

    input_ids = torch.tensor(
        encode(prompt),
        dtype=torch.long,
        device=device
    ).unsqueeze(0)

    generated_tokens = input_ids.squeeze(0).tolist()

    if strategy == "homeostasis":
        homeostasis = Homeostasis(
            decay=decay,
            increment=increment
        )

        for token in generated_tokens:
            homeostasis.update(token)

    with torch.no_grad():

        for _ in range(max_new_tokens):

            context = input_ids[:, -block_size:]
            outputs = model(input_ids=context)

            logits = outputs.logits[:, -1, :].squeeze(0)

            if strategy == "presence":

                logits = apply_presence_penalty(
                    logits,
                    generated_tokens,
                    alpha
                )

            elif strategy == "frequency":

                logits = apply_frequency_penalty(
                    logits,
                    generated_tokens,
                    beta
                )

            elif strategy == "homeostasis":

                homeostasis.decay_all()

                logits = apply_homeostatic_penalty(
                    logits,
                    homeostasis,
                    gamma
                )

            next_token = top_k_sampling(
                logits,
                k=top_k
            )

            generated_tokens.append(next_token)

            if strategy == "homeostasis":
                homeostasis.update(next_token)

            next_token_tensor = torch.tensor(
                [[next_token]],
                device=device
            )

            input_ids = torch.cat(
                [input_ids, next_token_tensor],
                dim=1
            )

    return decode(generated_tokens)

In [ ]:
# Baseline
print(generate(
    model,
    "ROMEO:",
    strategy="baseline"
))

ROMEO:
O, is some true, she'll not a man.

NORTHUMBERLAND:

Now I am to me say I be the king hath he should turn
And not so far as I will,
That God is the best to be thy son.

GLOUCESTER:
O, and it hath a gentleman, in all.

WARWICK:
So, my lord? how art not, thou speak.

BUCK


In [36]:
# Baseline Presence Penalty
print(generate(
    model,
    "ROMEO:",
    strategy="presence"
))

ROMEO:
O, sweetly come; but she be too for your hands! you.

JULIET:
Let, then my lord!

Nurse:
I think 'tis no harm. Let me have no more.

HORTENSIO:
To give them go, I can say you well;
Or else your high to keep good lady; and to go:
And I'll go so long as my son is not
And do it on


In [37]:
# Baseline Frequency Penalty
print(generate(
    model,
    "ROMEO:",
    strategy="frequency"
))

ROMEO:
I do: what is thy husband? and I hear thee good.

LADY ANNE:
You have you; if thou canst not live hence, and not speak?
My face is the queen, in such deep as I;
And there me too to my head, nor hope not so much too.

EDWARD:3 KING HENRY VI

GLOUCESTER:Hath hath nothing but the earth,
And will it


In [38]:
# Proposed Penalização Homeostática
print(generate(
    model,
    "ROMEO:",
    strategy="homeostasis"
))

ROMEO:
I cannot be: but then I know; yet that I'll speak.

SAMPSON:
Ay, and there not well as if it be, I must stand.

AUTOLYCUS:
'tis mine honour is so
Hanio? What duke is all, where, that it was but for she be gone.

CLAUDIO:
If I'll be good morrow now
I have said, sweetman, I am,


## Avaliação

In [ ]:
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

NUM_PROMPTS = 100
PROMPT_SIZE = 32
MAX_NEW_TOKENS = 512

In [40]:
prompts = []

for _ in range(NUM_PROMPTS):

    start = random.randint(
        0,
        len(test_data) - PROMPT_SIZE - 1
    )

    prompt_tokens = test_data[start:start + PROMPT_SIZE]

    prompt = tokenizer.decode(prompt_tokens.tolist())

    prompts.append(prompt)

print(f"{len(prompts)} prompts criados.")

100 prompts criados.


In [41]:
GENERATION_SEEDS = [0, 1, 2]

def generate_dataset(
    model,
    prompts,
    strategy,
    **kwargs
):

    generated = []

    total = len(prompts) * len(GENERATION_SEEDS)
    count = 0

    for i, prompt in enumerate(prompts):

        for seed in GENERATION_SEEDS:

            count += 1

            print(
                f"{strategy}: {count}/{total}",
                end="\r"
            )

            text = generate(
                model=model,
                prompt=prompt,
                strategy=strategy,
                seed=seed,
                max_new_tokens=MAX_NEW_TOKENS,
                **kwargs
            )

            generated.append({
                "prompt_id": i,
                "seed": seed,
                "prompt": prompt,
                "text": text
            })

    print()

    return generated

In [42]:
baseline_texts = generate_dataset(
    model,
    prompts,
    strategy="baseline",
    top_k=50
)

baseline: 300/300


In [43]:
presence_texts = generate_dataset(
    model,
    prompts,
    strategy="presence",
    top_k=50,
    alpha=0.5
)

presence: 300/300


In [44]:
frequency_texts = generate_dataset(
    model,
    prompts,
    strategy="frequency",
    top_k=50,
    beta=0.5
)

frequency: 300/300


In [45]:
homeostasis_texts = generate_dataset(
    model,
    prompts,
    strategy="homeostasis",
    top_k=50,
    gamma=0.5,
    decay=0.95,
    increment=1.0
)

homeostasis: 300/300


In [46]:
results = {
    "Baseline": baseline_texts,
    "Presence": presence_texts,
    "Frequency": frequency_texts,
    "Homeostasis": homeostasis_texts
}

In [47]:
with open("generated_texts.json", "w", encoding="utf-8") as f:

    json.dump(
        results,
        f,
        ensure_ascii=False,
        indent=4
    )

In [48]:
def distinct_n(text, n):

    tokens = encode(text)

    if len(tokens) < n:
        return 0.0

    ngrams = [
        tuple(tokens[i:i+n])
        for i in range(len(tokens)-n+1)
    ]

    unique = len(set(ngrams))

    total = len(ngrams)

    return unique / total


def distinct1(text):
    return distinct_n(text, 1)


def distinct2(text):
    return distinct_n(text, 2)

In [49]:
def unigram_repetition_rate(text):

    tokens = encode(text)

    if len(tokens) == 0:
        return 0.0

    return 1 - len(set(tokens)) / len(tokens)

def bigram_repetition_rate(text):

    tokens = encode(text)

    if len(tokens) < 2:
        return 0.0

    bigrams = [
        tuple(tokens[i:i+2])
        for i in range(len(tokens)-1)
    ]

    return 1 - len(set(bigrams)) / len(bigrams)

In [50]:
def shannon_entropy(text):

    tokens = encode(text)

    counts = Counter(tokens)

    probs = np.array(list(counts.values()), dtype=float)

    probs /= probs.sum()

    return -(probs * np.log2(probs)).sum()

In [51]:
bleu = BLEU(effective_order=True)

def self_bleu(texts):

    scores = []

    for i in range(len(texts)):

        hypothesis = texts[i]

        references = texts[:i] + texts[i+1:]

        score = bleu.corpus_score(
            [hypothesis],
            [references]
        ).score

        scores.append(score)

    return np.mean(scores)

In [52]:
def compute_perplexity(model, text, block_size, device):

    model.eval()

    tokens = torch.tensor(
        encode(text),
        dtype=torch.long
    )

    if len(tokens) < 2:
        return np.nan

    losses = []

    with torch.no_grad():

        for i in range(0, len(tokens)-1, block_size):

            chunk = tokens[i:i+block_size+1]

            if len(chunk) < 2:
                continue

            x = chunk[:-1].unsqueeze(0).to(device)
            y = chunk[1:].unsqueeze(0).to(device)

            outputs = model(input_ids=x)

            loss = F.cross_entropy(
                outputs.logits.reshape(-1, outputs.logits.size(-1)),
                y.reshape(-1)
            )

            losses.append(loss.item())

    return np.exp(np.mean(losses))

In [56]:
def evaluate_method(generations):

    metrics = {

        "Distinct-1": [],
        "Distinct-2": [],
        "Unigram Rep.": [],
        "Bigram Rep.": [],
        "Entropy": [],
        "Perplexity": []
        
    }

    all_texts = []

    for generation in generations:

        text = generation["text"]

        all_texts.append(text)

        metrics["Distinct-1"].append(distinct1(text))
        metrics["Distinct-2"].append(distinct2(text))
        metrics["Unigram Rep."].append(unigram_repetition_rate(text))
        metrics["Bigram Rep."].append(bigram_repetition_rate(text))
        metrics["Entropy"].append(shannon_entropy(text))
        metrics["Perplexity"].append(
            compute_perplexity(
                model,
                text,
                block_size,
                device
            )
        )

    metrics["Self-BLEU"] = self_bleu(all_texts)

    return metrics

In [57]:
def summarize_metrics(metrics):

    summary = {}

    for metric, values in metrics.items():

        if metric == "Self-BLEU":

            summary[metric] = values

        else:

            summary[metric] = {

                "mean": np.nanmean(values),
                "std": np.nanstd(values)
            }

    return summary

In [58]:
results_table = {}

for method, texts in results.items():

    print(f"Avaliando {method}...")

    results_table[method] = evaluate_method(texts)

Avaliando Baseline...
Avaliando Presence...
Avaliando Frequency...
Avaliando Homeostasis...


In [59]:
df = pd.DataFrame(results_table).T

display(df)

,Distinct-1,Distinct-2,Unigram Rep.,Bigram Rep.,Entropy,Perplexity,Self-BLEU
Baseline,"[0.3474264705882353, 0.3639705882352941, 0.383...","[0.7219152854511971, 0.7348066298342542, 0.747...","[0.6525735294117647, 0.6360294117647058, 0.616...","[0.2780847145488029, 0.26519337016574585, 0.25...","[6.561385569301047, 6.507091687679306, 6.59964...","[17.70908099390459, 12.512201193128421, 15.141...",6.128632
Presence,"[0.4143646408839779, 0.43014705882352944, 0.44...","[0.7896678966789668, 0.8158379373848987, 0.821...","[0.5856353591160222, 0.5698529411764706, 0.555...","[0.21033210332103325, 0.18416206261510126, 0.1...","[6.670185246028177, 6.837041743513313, 6.85979...","[18.668743460569875, 24.479037397101756, 23.98...",4.253124
Frequency,"[0.5185873605947955, 0.5276752767527675, 0.512...","[0.9664804469273743, 0.9593345656192237, 0.938...","[0.4814126394052045, 0.4723247232472325, 0.487...","[0.03351955307262566, 0.0406654343807763, 0.06...","[7.704081867664788, 7.685094894800211, 7.65463...","[167.1274388084744, 88.67779566060025, 106.543...",1.928617
Homeostasis,"[0.4014732965009208, 0.3786764705882353, 0.373...","[0.8154981549815498, 0.7863720073664825, 0.809...","[0.5985267034990792, 0.6213235294117647, 0.626...","[0.18450184501845024, 0.2136279926335175, 0.19...","[6.782822972114245, 6.666619848039924, 6.65926...","[21.804783734556384, 17.54819399520602, 23.753...",3.736728


In [60]:
summary_table = {}

for method in results_table:

    summary_table[method] = summarize_metrics(
        results_table[method]
    )

In [61]:
df2 = pd.DataFrame(summary_table).T

display(df2)

,Distinct-1,Distinct-2,Unigram Rep.,Bigram Rep.,Entropy,Perplexity,Self-BLEU
Baseline,"{'mean': 0.37874109674961043, 'std': 0.0180155...","{'mean': 0.767324557885352, 'std': 0.026550870...","{'mean': 0.6212589032503896, 'std': 0.01801557...","{'mean': 0.2326754421146479, 'std': 0.02655087...","{'mean': 6.597730810241046, 'std': 0.100947476...","{'mean': 20.410875941244466, 'std': 3.79111438...",6.128632
Presence,"{'mean': 0.4327378389706024, 'std': 0.01683381...","{'mean': 0.8120460771221286, 'std': 0.02177092...","{'mean': 0.5672621610293976, 'std': 0.01683381...","{'mean': 0.18795392287787133, 'std': 0.0217709...","{'mean': 6.822707087640594, 'std': 0.090861831...","{'mean': 22.794607691911914, 'std': 4.00635806...",4.253124
Frequency,"{'mean': 0.5064816799862274, 'std': 0.01365045...","{'mean': 0.9550515836272143, 'std': 0.00822256...","{'mean': 0.4935183200137726, 'std': 0.01365045...","{'mean': 0.044948416372785654, 'std': 0.008222...","{'mean': 7.660861099087327, 'std': 0.041334961...","{'mean': 115.2715762113732, 'std': 16.99292891...",1.928617
Homeostasis,"{'mean': 0.3898262696683875, 'std': 0.01647707...","{'mean': 0.7992416121927084, 'std': 0.02236800...","{'mean': 0.6101737303316125, 'std': 0.01647707...","{'mean': 0.20075838780729166, 'std': 0.0223680...","{'mean': 6.7428780973961135, 'std': 0.07822226...","{'mean': 23.364247229074497, 'std': 4.10996009...",3.736728


In [62]:
def cohens_d_paired(x, y):

    x = np.asarray(x)
    y = np.asarray(y)

    diff = x - y

    return diff.mean() / diff.std(ddof=1)

In [ ]:
def compare_methods(
    metric_name,
    method1,
    method2,
    alpha=0.05
):

    x = np.asarray(results_table[method1][metric_name])
    y = np.asarray(results_table[method2][metric_name])

    mask = ~(np.isnan(x) | np.isnan(y))

    x = x[mask]
    y = y[mask]

    differences = x - y

    p_normality = shapiro(differences).pvalue

    if p_normality >= alpha:

        statistic, p_value = ttest_rel(x, y)

        test = "Paired t-test"

    else:

        statistic, p_value = wilcoxon(x, y)

        test = "Wilcoxon"

    effect = cohens_d_paired(x, y)

    return {

        "Metric": metric_name,

        "Method A": method1,

        "Method B": method2,

        "Test": test,

        "Statistic": statistic,

        "p-value": p_value,

        "Effect Size": effect

    }

In [64]:
comparisons = []

metrics = [

    "Distinct-1",

    "Distinct-2",

    "Unigram Rep.",

    "Bigram Rep.",

    "Entropy",

    "Perplexity"

]

pairs = [

    ("Baseline", "Homeostasis"),

    ("Presence", "Homeostasis"),

    ("Frequency", "Homeostasis")

]

for metric in metrics:

    for method_a, method_b in pairs:

        comparisons.append(

            compare_methods(

                metric,

                method_a,

                method_b

            )

        )

In [65]:
stats_df = pd.DataFrame(comparisons)

display(stats_df)

,Metric,Method A,Method B,Test,Statistic,p-value,Effect Size
0,Distinct-1,Baseline,Homeostasis,Paired t-test,-8.123092,1.206814e-14,-0.468987
1,Distinct-1,Presence,Homeostasis,Paired t-test,34.699181,7.372084e-107,2.003358
2,Distinct-1,Frequency,Homeostasis,Paired t-test,97.657954,7.184083e-229,5.638285
3,Distinct-2,Baseline,Homeostasis,Paired t-test,-16.514022,5.465642e-44,-0.953437
4,Distinct-2,Presence,Homeostasis,Wilcoxon,11362.500000,1.430016e-13,0.440339
5,Distinct-2,Frequency,Homeostasis,Paired t-test,115.144713,1.050694e-249,6.647883
6,Unigram Rep.,Baseline,Homeostasis,Paired t-test,8.123092,1.206814e-14,0.468987
7,Unigram Rep.,Presence,Homeostasis,Paired t-test,-34.699181,7.372084e-107,-2.003358
8,Unigram Rep.,Frequency,Homeostasis,Paired t-test,-97.657954,7.184083e-229,-5.638285
9,Bigram Rep.,Baseline,Homeostasis,Paired t-test,16.514022,5.465642e-44,0.953437
